In [ ]:
%%bash 

curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
%pip install -U crewai crewai_tools 'crewai[tools, litellm]' unstructured

##### Run ollama locally

In [ ]:
%%bash 
curl -fsSL https://ollama.com/install.sh | sh

#
ollama list

#
ollama run llama3.2 
ollama stop llama3.2

#
sudo systemctl start ollama
sudo systemctl stop ollama


# embeddings
ollama pull all-minilm
ollama run all-minilm
curl http://localhost:11434/api/embeddings -d '{
  "model": "all-minilm",
  "prompt": "The sky is blue because of Rayleigh scattering"
}'


#
curl -X POST http://localhost:11434/api/generate -d '{
  "model": "llama3",
  "prompt":"Why is the sky blue?"
}'

# ollama run deepseek-r1:8b
# ollama run deepseek-r1:14b
# ollama run deepseek-r1:32b
# ollama run hf.co/brijeshdhaker/gpt-oss-20b-GGUF:Q4_K_M
# http://localhost:11434
# gpt-oss-120b
ollama pull hf.co/openai/gpt-oss-20b
ollama run hf.co/openai/gpt-oss-20b


ollama rm llama3
# /usr/share/ollama/.ollama/models

curl -X DELETE http://localhost:11434/api/delete -d '{ "model": "gemma3" }'

#### Seup Docker Model Runner Locally

In [ ]:
%%bash

sudo -S apt-get update
sudo -S apt-get install docker-model-plugin

%%bash

docker model pull ai/smollm2:360M-Q4_K_M
docker model pull hf.co/brijeshdhaker/Llama-3.2-1B-Instruct-GGUF

In [ ]:
%%bash
docker model reinstall-runner --backend vllm --gpu cuda

docker model pull ai/gpt-oss:20B
docker model run --detach ai/gpt-oss:20B
docker model stop ai/gpt-oss:20B

docker model pull ai/smollm2
docker model run --detach ai/smollm2
docker model stop ai/smollm2

# Pull a small, efficient model (good for getting started)
docker model pull ai/llama3.2:1B-Q8_0
docker model run --detach ai/llama3.2:1B-Q8_0

# Pull a larger, more capable model
docker model pull ai/llama3.2:3B-Q4_K_M

# Pull a code-specialized model
docker model pull ai/codellama:7B-Q4_K_M

docker model pull ai/llama3.2
docker model pull ai/llama3.2:1B-Q8_0
docker model run --detach ai/llama3.2
docker model stop ai/llama3.2

# List all downloaded models with sizes
docker model list

# Show detailed information about a model
docker model inspect ai/llama3.2:1B-Q8_0

# Remove a model to free disk space
docker model rm ai/llama3.2:1B-Q8_0

# Remove all unused models
docker model prune

docker model uninstall-runner --models
docker model uninstall-runner --images

http://localhost:12434/engines/v1/models

http://vmware-ubuntu.sandbox.net:12434
# The model is now serving on a local endpoint
# Default: http://localhost:12434/v1

#
curl http://localhost:12434/engines/v1/embeddings \
  -H "Content-Type: application/json" \
  -d '{
    "model": "ai/qwen3-embedding",
    "input": "A dog is an animal"
  }'
  
curl http://localhost:12434/api/chat \
  -H "Content-Type: application/json" \
  -d '{"model": "ai/smollm2", "messages": [{"role": "user", "content": "Hello!"}]}'

# Send a chat completion request using curl
curl http://localhost:12434/engines/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "ai/llama3.2:1B-Q8_0",
    "messages": [
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "Explain Docker volumes in 3 sentences."}
    ],
    "temperature": 0.7,
    "max_tokens": 256
  }'

In [ ]:
%mkdir -p agentic-ai/crewai/config

```yaml
# ./docker-compose.yaml
services:
  app:
    image: my-app
    models:
      llm:
        endpoint_var: AI_MODEL_URL
        model_var: AI_MODEL_NAME
      embedding-model:
        endpoint_var: EMBEDDING_URL
        model_var: EMBEDDING_NAME

models:
  dev_model:
    model: ai/model
    context_size: 4096
    runtime_flags:
  llm:
    model: ai/smollm2
    context_size: 4096
    runtime_flags:
      - "--temp 0.1"            # Temperature
      - "--top-p 0.9"           # Top-p sampling
      - "--verbose"             # Set verbosity level to infinity
      - "--verbose-prompt"      # Print a verbose prompt before generation
      - "--log-prefix"          # Enable prefix in log messages
      - "--log-timestamps"      # Enable timestamps in log messages
      - "--log-colors"          # Enable colored logging
  embedding-model:
    model: ai/all-minilm
    context_size: 2048
    runtime_flags:
      - "--embeddings"
```

In [ ]:
%%bash

pip install crewai crewai_tools langchain langchain_community langchain_ollama streamlit duckduckgo-search

In [ ]:
%%bash

cat <<EOF > agentic-ai/crewai/config/agents.yaml
researcher:
  role: >
    {topic} Senior Data Researcher
  goal: >
    Uncover cutting-edge developments in {topic}
  backstory: >
    You're a seasoned researcher with a knack for uncovering the latest
    developments in {topic}. You find the most relevant information and
    present it clearly.
EOF


In [ ]:
%%bash

cat <<EOF > agentic-ai/crewai/config/tasks.yaml
research_task:
  description: >
    Conduct thorough research about {topic}. Use web search to find current,
    credible information. The current year is 2026.
  expected_output: >
    A markdown report with clear sections: key trends, notable tools or companies,
    and implications. Aim for 800–1200 words. No fenced code blocks around the whole document.
  agent: researcher
  output_file: output/report.md
EOF

In [ ]:
# src/latest_ai_flow/crews/content_crew/content_crew.py
from typing import List

from crewai import Agent, Crew, Process, Task
from crewai.agents.agent_builder.base_agent import BaseAgent
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool


@CrewBase
class ResearchCrew:
  """Single-agent research crew used inside the Flow."""

  agents: List[BaseAgent]
  tasks: List[Task]

  agents_config = "agentic-ai/crewai/config/agents.yaml"
  tasks_config = "agentic-ai/crewai/config/tasks.yaml"

  @agent
  def researcher(self) -> Agent:
    return Agent(
      config=self.agents_config["researcher"],  # type: ignore[index]
      verbose=True,
      tools=[SerperDevTool()],
    )

  @task
  def research_task(self) -> Task:
    return Task(
      config=self.tasks_config["research_task"],  # type: ignore[index]
    )

  @crew
  def crew(self) -> Crew:
    return Crew(
      agents=self.agents,
      tasks=self.tasks,
      process=Process.sequential,
      verbose=True,
    )

In [ ]:
# src/latest_ai_flow/main.py
from pydantic import BaseModel
from crewai.flow import Flow, listen, start
#from latest_ai_flow.crews.content_crew.content_crew import ResearchCrew


class ResearchFlowState(BaseModel):
  topic: str = ""
  report: str = ""


class LatestAiFlow(Flow[ResearchFlowState]):
  @start()
  def prepare_topic(self, crewai_trigger_payload: dict | None = None):
    if crewai_trigger_payload:
      self.state.topic = crewai_trigger_payload.get("topic", "AI Agents")
    else:
      self.state.topic = "AI Agents"
    print(f"Topic: {self.state.topic}")

  @listen(prepare_topic)
  def run_research(self):
    result = ResearchCrew().crew().kickoff(inputs={"topic": self.state.topic})
    self.state.report = result.raw
    print("Research crew finished.")

  @listen(run_research)
  def summarize(self):
    print("Report path: output/report.md")


def kickoff():
  LatestAiFlow().kickoff()


def plot():
  LatestAiFlow().plot()


#if __name__ == "__main__":
#  kickoff()

In [ ]:
kickoff()

In [ ]:
from crewai import LLM
from crewai import Agent, Task, Crew
from langchain_ollama.llms import OllamaLLM
from crewai import Crew, Process

# Set up Ollama LLM
# ollama_llm = LLM(
#    model="ollama/llama3.2",
#    base_url="http://localhost:11434"
# )

docker_llm = LLM(
    model="openai/ai/llama3.2:1B-Q8_0",
    base_url="http://localhost:12434/engines/v1"
)




# Define your agents
researcher = Agent(
    role='Senior Research Analyst',
    goal='Discover new insights',
    backstory="""You're an expert at finding interesting information""",
    llm=ollama_llm,
    verbose=True
)

writer = Agent(
    role='Content Writer',
    goal='Write engaging content',
    backstory="""You're a talented writer who simplifies complex information""",
    llm=ollama_llm,
    verbose=True
)

# Create tasks
research_task = Task(
    description='Find interesting facts about AI in healthcare',
    expected_output="A clear, concise summary about AI in healthcare",
    agent=researcher
)

write_task = Task(
    description='Write a short blog post about AI in healthcare',
    expected_output="Write a short blog post about AI in healthcare",
    agent=writer
)

# Form the crew
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    verbose=True
)

# Execute the crew's tasks
result = crew.kickoff()

print("Here's the result:")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4f62311e-18e8-4e47-9bfe-82bb07216e0f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find interesting facts about AI in healthcare                                                            │
│  ID: 858bae65-952c-40b3-8868-b47306ddd1fe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Find interesting facts about AI in healthcare                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Artificial Intelligence in Healthcare: Revolutionizing Patient Care**                                        │
│                                                                                                                 │
│  Artificial intelligence (AI) is transforming the healthcare industry by enabling the development of            │
│  personalized, efficient, and cost-effective medical treatments. Here are some interesting facts about AI in    │
│  healthcare:                                                                                                    │
│                                                                                                                 │
│  1. **Predictive Maintenance**: AI-powered sensors and machine learning algorithms can detect anomalies in      │
│  medical equipment, predict when maintenance is required, and notify healthcare professionals to prevent        │
│  malfunctions.                                                                                                  │
│  2. **Disease Diagnosis**: AI-assisted diagnostic tools can analyze medical images, patient data, and clinical  │
│  notes to identify diseases such as breast cancer, lung nodules, and diabetic retinopathy with high accuracy.   │
│  3. **Personalized Medicine**: AI can help tailor treatment plans to individual patients based on their         │
│  genetic profiles, medical histories, and lifestyle factors.                                                    │
│  4. **Virtual Assistants**: AI-powered chatbots and virtual assistants, such as IBM's Watson for Oncology, can  │
│  assist patients with routine queries, medication reminders, and symptom check-ups.                             │
│  5. **Robotic Surgery**: AI-powered robots can assist surgeons during operations, enhancing precision,          │
│  reducing recovery time, and improving patient outcomes.                                                        │
│  6. **Predictive Analytics**: AI can analyze large datasets to predict patient outcomes, identify high-risk     │
│  patients, and detect potential medical emergencies before they occur.                                          │
│  7. **Imaging Analysis**: AI can automatically analyze medical images, such as CT scans and MRI scans, to       │
│  detect conditions like tumors, fractures, and cardiovascular disease.                                          │
│  8. **Telemedicine**: AI-powered telemedicine platforms can connect patients with healthcare professionals      │
│  remotely, enabling early diagnosis, treatment, and preventive care.                                            │
│  9. **Automated Clinical Documentation**: AI can help automate clinical documentation, reducing errors, and     │
│  freeing up healthcare professionals to focus on patient care.                                                  │
│  10. **Research and Development**: AI can accelerate the research and development process, enabling the         │
│  discovery of new treatments and therapies by analyzing vast amounts of medical data.                           │
│                                                                                                                 │
│  **Real-World Applications**:                                                                                   │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Find interesting facts about AI in healthcare                                                            │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Artificial Intelligence in Healthcare: Revolutionizing Patient Care**                                        │
│                                                                                                                 │
│  Artificial intelligence (AI) is transforming the healthcare industry by enabling the development of            │
│  personalized, efficient, and cost-effective medical treatments. The integration of AI in healthcare has the    │
│  potential to revolutionize the way patients are diagnosed, treated, and managed, leading to improved patient   │
│  outcomes, enhanced care experiences, and increased efficiency.                                                 │
│                                                                                                                 │
│  **Predictive Maintenance: The Future of Healthcare Equipment**                                                 │
│                                                                                                                 │
│  Artificial intelligence-powered sensors and machine learning algorithms are revolutionizing the way medical    │
│  equipment is maintained. These sensors can detect anomalies in equipment, predict when maintenance is          │
│  required, and notify healthcare professionals to prevent malfunctions. For example, AI-powered sensors can     │
│  monitor the temperature of medical equipment, alerting healthcare professionals when it reaches a critical     │
│  level. This proactive approach to maintenance can help prevent costly repairs, reduce downtime, and improve    │
│  patient safety.                                                                                                │
│                                                                                                                 │
│  **Disease Diagnosis: AI-Powered Diagnostic Tools**                                                             │
│                                                                                                                 │
│  AI-assisted diagnostic tools are transforming the way diseases are diagnosed. These tools can analyze medical  │
│  images, patient data, and clinical notes to identify diseases such as breast cancer, lung nodules, and         │
│  diabetic retinopathy with high accuracy. For instance, AI-powered imaging analysis can detect breast cancer    │
│  by analyzing breast tissue samples, identifying tumors, and detecting cancerous cells. Similarly, AI-assisted  │
│  diagnostic tools can analyze patient data to identify diabetic retinopathy, detecting high-risk patients and   │
│  enabling early intervention.                                                                                   │
│                                                                                                                 │
│  **Personalized Medicine: Tailoring Treatment Plans**                                                           │
│                                                                                                                 │
│  AI can help tailor treatment plans to individual patients based on their genetic profiles, medical histories,  │
│  and lifestyle factors. For example, AI can analyze genetic data to identify patients at high risk of           │
│  developing certain diseases, enabling early interventi

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short blog post about AI in healthcare                                                           │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Here's the result:
**Artificial Intelligence in Healthcare: Revolutionizing Patient Care**

Artificial intelligence (AI) is transforming the healthcare industry by enabling the development of personalized, efficient, and cost-effective medical treatments. The integration of AI in healthcare has the potential to revolutionize the way patients are diagnosed, treated, and managed, leading to improved patient outcomes, enhanced care experiences, and increased efficiency.

**Predictive Maintenance: The Future of Healthcare Equipment**

Artificial intelligence-powered sensors and machine learning algorithms are revolutionizing the way medical equipment is maintained. These sensors can detect anomalies in equipment, predict when maintenance is required, and notify healthcare professionals to prevent malfunctions. For example, AI-powered sensors can monitor the temperature of medical equipment, alerting healthcare professionals when it reaches a critical level. This proactive approach to maint

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ d06f310a-50c4-4625-8e46-1e5ee5918e94                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/d06f310a-50c4-462 │
│ 5-8e46-1e5ee5918e94?access_code=TRACE-e356516340                             │
│ 🔑 Access Code: TRACE-e356516340                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4f62311e-18e8-4e47-9bfe-82bb07216e0f                                                                       │
│  Final Output: **Artificial Intelligence in Healthcare: Revolutionizing Patient Care**                          │
│                                                                                                                 │
│  Artificial intelligence (AI) is transforming the healthcare industry by enabling the development of            │
│  personalized, efficient, and cost-effective medical treatments. The integration of AI in healthcare has the    │
│  potential to revolutionize the way patients are diagnosed, treated, and managed, leading to improved patient   │
│  outcomes, enhanced care experiences, and increased efficiency.                                                 │
│                                                                                                                 │
│  **Predictive Maintenance: The Future of Healthcare Equipment**                                                 │
│                                                                                                                 │
│  Artificial intelligence-powered sensors and machine learning algorithms are revolutionizing the way medical    │
│  equipment is maintained. These sensors can detect anomalies in equipment, predict when maintenance is          │
│  required, and notify healthcare professionals to prevent malfunctions. For example, AI-powered sensors can     │
│  monitor the temperature of medical equipment, alerting healthcare professionals when it reaches a critical     │
│  level. This proactive approach to maintenance can help prevent costly repairs, reduce downtime, and improve    │
│  patient safety.                                                                                                │
│                                                                                                                 │
│  **Disease Diagnosis: AI-Powered Diagnostic Tools**                                                             │
│                                                                                                                 │
│  AI-assisted diagnostic tools are transforming the way diseases are diagnosed. These tools can analyze medical  │
│  images, patient data, and clinical notes to identify diseases such as breast cancer, lung nodules, and         │
│  diabetic retinopathy with high accuracy. For instance, AI-powered imaging analysis can detect breast cancer    │
│  by analyzing breast tissue samples, identifying tumors, and detecting cancerous cells. Similarly, AI-assisted  │
│  diagnostic tools can analyze patient data to identify diabetic retinopathy, detecting high-risk patients and   │
│  enabling early intervention.                                                                                   │
│                                                                                                                 │
│  **Personalized Medicine: Tailoring Treatment Plans**                                                           │
│                                                                                                                 │
│  AI can help tailor treatment plans to individual patients based on their genetic profiles, medical histories,  │
│  and lifestyle factors. For example, AI can analyze genetic data to identify patients at high risk of           │
│  developing certain diseases, enabling early intervent